## Jupyter kernels from the Qiskit container (OnDemand)

This notebook registers two Jupyter kernels that run Python **inside** the
`qiskit.sif` Apptainer container, so the other notebooks in this folder get
Qiskit 2.5, the addons and the GPU build of Qiskit Aer without any local
installation:

| kernel | Apptainer flags | use |
|---|---|---|
| **Python (Qiskit CPU)** | none | any node |
| **Python (Qiskit GPU)** | `--nv` (exposes the host NVIDIA driver) | GPU nodes, for `AerSimulator(device="GPU")` |

Run it **once**, with the default **Python 3 (ipykernel)** kernel selected in
the top-right corner. The kernels are saved in
`~/.local/share/jupyter/kernels/` and stay available in later sessions.

##### 1. Check that the container starts and has `ipykernel` and `qiskit`

In [ ]:
# `!` runs a shell command.  `apptainer exec` starts the container and runs
# python3 inside it; ipykernel is what lets Jupyter talk to that Python.
!apptainer exec --cleanenv /ocean/containers/quantum/qiskit.sif python3 -c "import ipykernel, qiskit, qiskit_aer; print('ipykernel', ipykernel.__version__, '| qiskit', qiskit.__version__, '| qiskit-aer', qiskit_aer.__version__)"

##### 2. Register the CPU and GPU kernels

In [ ]:
import json, os, shutil   # standard library

sif    = "/ocean/containers/quantum/qiskit.sif"
runner = shutil.which("apptainer")           # full path of the apptainer command
assert runner and os.path.exists(sif), "apptainer or the container image is missing"

for name, display, flags in [("qiskit-cpu", "Python (Qiskit CPU)", []),
                             ("qiskit-gpu", "Python (Qiskit GPU)", ["--nv"])]:
    # Jupyter finds user kernels in ~/.local/share/jupyter/kernels/<name>/kernel.json
    kdir = os.path.expanduser(f"~/.local/share/jupyter/kernels/{name}")
    os.makedirs(kdir, exist_ok=True)
    # The command Jupyter runs to start the kernel:
    #   --cleanenv            do not pass the host environment (OnDemand's Python
    #                         paths) into the container
    #   --nv                  GPU kernel only: make the host NVIDIA driver visible
    #   --bind /ocean         make project and container folders readable/writable
    #   PYTHONNOUSERSITE=1    ignore packages in ~/.local, use the container's only
    #   {connection_file}     Jupyter fills this in with the kernel's port file
    argv = [runner, "exec", "--cleanenv", *flags,
            "--bind", "/ocean",
            "--env", "PYTHONNOUSERSITE=1",
            sif, "python3", "-m", "ipykernel_launcher", "-f", "{connection_file}"]
    with open(f"{kdir}/kernel.json", "w") as f:
        json.dump({"argv": argv, "display_name": display, "language": "python"}, f, indent=2)
    print("created", name, "->", kdir)

From now on, select the kernel in the top-right corner (refresh the page if
it is not listed yet): **Python (Qiskit CPU)** on a CPU node, or
**Python (Qiskit GPU)** on a GPU node.

> **Caution:** `--cleanenv` also removes the Slurm variables, so the kernel
> does not know how many cores your job has. On a shared node, limit Aer's
> threads to your allocation, e.g. `AerSimulator(max_parallel_threads=8)`,
> or set `os.environ["OMP_NUM_THREADS"] = "8"` before importing Qiskit.

##### 3. Check the new kernel

Switch to **Python (Qiskit GPU)** (or CPU), then run the cell below.
`available_devices()` lists what Aer was compiled for; the 2-qubit run on
`device="GPU"` only succeeds if a GPU is actually visible.

In [ ]:
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

print("qiskit", qiskit.__version__)
print("Aer compiled for:", AerSimulator().available_devices())

# Bell state: expect about half '00' and half '11'
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

for device in ("CPU", "GPU"):
    try:
        counts = AerSimulator(device=device).run(qc, shots=1000).result().get_counts()
        print(f"{device}: {counts}")
    except Exception as exc:          # no GPU visible (CPU kernel, or CPU node)
        print(f"{device}: not available ({type(exc).__name__})")